<a href="https://colab.research.google.com/github/Madankk-06/Deep-learning-projects/blob/main/7_ResNet_50_Style_Residual_Bottleneck_Block_and_Grouped_Convolutionsipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Import PyTorch for implementing custom neural network layers.
import torch

# Import neural network modules.
import torch.nn as nn

# Import TorchVision for model-related utilities.
import torchvision

# Import PyTorch Profiler to analyze execution time,
# memory consumption, and computational complexity.
from torch.profiler import profile, ProfilerActivity

In [2]:
# Implement Depthwise Separable Convolution.
# The operation separates spatial filtering and
# channel mixing to reduce computation and parameters.

class DepthwiseSeparableConv(nn.Module):

    def __init__(self, in_channels, out_channels, stride=1):

        super().__init__()

        # Perform spatial convolution independently
        # on each input channel.
        self.depthwise = nn.Conv2d(
            in_channels,
            in_channels,
            kernel_size=3,
            stride=stride,
            padding=1,
            groups=in_channels,
            bias=False
        )

        # Combine information across channels
        # using Pointwise (1×1) convolution.
        self.pointwise = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=1,
            bias=False
        )

    def forward(self, x):

        x = self.depthwise(x)

        x = self.pointwise(x)

        return x

In [3]:
# Implement a ResNet-50 Bottleneck Block.
# The bottleneck structure reduces computation
# while preserving feature representation.

class BottleneckBlock(nn.Module):

    def __init__(self, in_channels, out_channels, stride=1):

        super().__init__()

        # Reduce feature dimensions using 1×1 convolution.
        self.conv1 = nn.Conv2d(
            in_channels,
            out_channels // 4,
            kernel_size=1,
            bias=False
        )

        # Perform efficient spatial feature extraction
        # using Depthwise Separable Convolution.
        self.conv2 = DepthwiseSeparableConv(
            out_channels // 4,
            out_channels // 4,
            stride
        )

        # Restore channel dimensions using 1×1 convolution.
        self.conv3 = nn.Conv2d(
            out_channels // 4,
            out_channels,
            kernel_size=1,
            bias=False
        )

        # Learnable projection aligns feature dimensions
        # before residual addition.
        self.projection = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=1,
            stride=stride,
            bias=False
        )

        self.relu = nn.ReLU()

    def forward(self, x):

        # Preserve the shortcut connection.
        identity = self.projection(x)

        out = self.conv1(x)

        out = self.relu(out)

        out = self.conv2(out)

        out = self.relu(out)

        out = self.conv3(out)

        # Perform residual learning by combining
        # transformed features with shortcut features.
        out = out + identity

        out = self.relu(out)

        return out

In [4]:
# Create a ResNet Bottleneck Block
# for feature extraction.

model = BottleneckBlock(
    in_channels=64,
    out_channels=256,
    stride=1
)

In [5]:
# Generate a batch of feature maps
# for testing the bottleneck block.

input_tensor = torch.randn(
    8,
    64,
    56,
    56
)

In [6]:
# Perform forward propagation through
# the custom ResNet Bottleneck Block.

output = model(input_tensor)

print("Output Shape :", output.shape)

Output Shape : torch.Size([8, 256, 56, 56])


In [7]:
# Calculate the total number of learnable
# parameters in the bottleneck architecture.

total_parameters = sum(

    parameter.numel()

    for parameter in model.parameters()

)

print("Total Parameters :", total_parameters)

Total Parameters : 41536


In [8]:
# Profile execution time, memory usage,
# and computational performance of the model.

with profile(

    activities=[
        ProfilerActivity.CPU
    ],

    profile_memory=True,

    record_shapes=True

) as profiler:

    output = model(input_tensor)

# Display execution statistics collected
# by the profiler.

print(

    profiler.key_averages().table(

        sort_by="cpu_time_total"

    )

)

------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                          Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg       CPU Mem  Self CPU Mem    # of Calls  
------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                  aten::conv2d         0.17%     245.582us        67.15%      97.890ms      19.578ms      67.38 MB           0 B             5  
             aten::convolution         0.13%     193.376us        66.98%      97.644ms      19.529ms      67.38 MB           0 B             5  
            aten::_convolution         0.09%     134.227us        66.85%      97.451ms      19.490ms      67.38 MB           0 B             5  
             aten::thnn_conv2d         0.03%      43.272us        57.42%      83.707ms      20.927ms      61.25 MB           0 B  

/usr/local/lib/python3.12/dist-packages/torch/profiler/profiler.py:224: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(


In [9]:
# Compare parameter growth under
# different channel scaling factors.

for channels in [64,128,256,512]:

    model = BottleneckBlock(

        in_channels=channels,

        out_channels=channels*4

    )

    parameters = sum(

        parameter.numel()

        for parameter in model.parameters()

    )

    print(

        f"Channels : {channels} | Parameters : {parameters}"

    )

Channels : 64 | Parameters : 41536
Channels : 128 | Parameters : 164992
Channels : 256 | Parameters : 657664
Channels : 512 | Parameters : 2626048


In [10]:
# Estimate computational complexity by
# observing profiler statistics under
# different channel configurations.

print(

    "Profiler statistics can be used to compare",

    "execution cost, memory usage, and",

    "floating-point operations across",

    "different scaling factors."

)

Profiler statistics can be used to compare execution cost, memory usage, and floating-point operations across different scaling factors.
